In [ ]:
import os
import csv
import pickle
import logging
import numpy as np
import pandas as pd
import sumolib
from datetime import datetime 
import xml.etree.ElementTree as ET
from scipy.stats import wasserstein_distance
from typing import List, Union
from matplotlib import pyplot as plt
from collections import defaultdict
from scipy.interpolate import LinearNDInterpolator
import time
import ijson
import seaborn as sns
import sys
sys.path.append("..")
from benchmark.error_funcs import VelocityGridErrorCalculator as Error

In [ ]:
from benchmark.error_funcs import MicroscopicErrorCalculator as Error

In [ ]:
with open("timestamps.pkl", "rb") as f:
    timestamps = pickle.load(f)

print(timestamps)

In [ ]:
inception_df = pd.read_csv("error/inception_1130_westbound_traj_data.csv")
inception_df["timestamp"] = pd.to_datetime(inception_df["timestamp"])


velocity_grid_ec = Error(net_file_path="../../sim_files/sumo_test.net.xml",
                                            inception_file_path="../i24_data/11-30.json",
                                            target_times=timestamps, 
                                            granularity = 1.0)

flow_hat, density_hat = velocity_grid_ec.get_flow_density_matrix(inception_df,
                                                                  lane_func=velocity_grid_ec.westbound_lane_func,
                                                                  time_interval=pd.Timedelta(seconds=10),
                                                                  space_interval=400,
                                                                  output_filename="wb_flow_density_hat.csv")

pd.DataFrame(flow_hat).to_csv("error/wb_flow_hat_raw.csv")
pd.DataFrame(density_hat).to_csv("error/wb_density_hat_raw.csv")

flow_hat = flow_hat[:,:-1]
flow_hat = np.flip(flow_hat, axis=1)
            
density_hat = density_hat[:,:-1]
density_hat = np.flip(density_hat, axis=1)

velocity_hat = flow_hat/density_hat
print("shape of vel_hat:", velocity_hat.shape)
print(velocity_hat)

np.save("error/wb_flow_hat.npy", flow_hat)
np.save("error/wb_density_hat.npy", density_hat)
np.save("error/wb_velocity_hat.npy", velocity_hat)

pd.DataFrame(velocity_hat).to_csv("error/wb_velocity_hat.csv")
pd.DataFrame(flow_hat).to_csv("error/wb_flow_hat.csv")
pd.DataFrame(density_hat).to_csv("error/wb_density_hat.csv")

In [ ]:
df = pd.read_csv("intermediate_output2.csv")
count = (df["space_bin"] == 2).sum()

print("Number of rows:", count)

In [ ]:
v_pred = vel_grid
vmax = print(np.max(vel_grid))
plt.figure(figsize=(10, 6))
plt.imshow(v_pred.T, aspect='auto', origin='lower', cmap='RdYlGn', interpolation="none", vmin = 0, vmax = 120)

# Label axes
plt.xlabel('Time Step (10s intervals)')
plt.ylabel('Segment Index (400m intervals)')
plt.title('Time-Space Diagram of Predicted Velocities - GA Optimized (workers = 2, num_generations = 1, sol_per_pop = 2), MAPE = 181.67')
plt.colorbar(label='Velocity (km/hr)')

plt.tight_layout()
# plt.show()

In [ ]:
micro_calc = Error(
    micro_obj_fn_timestamps=timestamps,
    fcd_xml_file_path="../sim_files/scen3_unnoised_1130_0420-0480/0/fcd_output.xml",
    net_file_path="../../sim_files/sumo_test.net.xml"
)

vel_grid, t_bins, x_bins, vel_df = micro_calc.compute_velocity_grid_from_fcd(
    time_interval=pd.Timedelta(seconds=10),
    space_interval=400,
    x_min_miles=58.8,
    x_max_miles=62.8,
    return_df=True
)

print(vel_df.head())


print("Velocity grid shape:", vel_grid.shape)
print("Units: km/h")